In [ ]:
import os
import sys

# Make the project's src package available from the notebooks folder.
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, wearables, baselines, evaluation

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
SEED = 0

In [ ]:
# Modelling setup: same target, cycle rules and participant split as 02_baselines.
TARGET = "days_until_next_period"
# TODO(INTERVAL2_REVIEW): Expand only after the authors' response is reviewed.
# Recheck HR aggregation in src/wearables.py and Interval 2 phase-derived labels.
MODEL_INTERVALS = (2022,)

baseline_diary = data.load_diary()
baseline_starts = data.period_starts(baseline_diary, gap_days=3)
baseline_cycles = data.build_cycles(baseline_starts, min_len=15, max_len=60)

# Split BEFORE restricting intervals: preserve baseline participant assignments.
model_split = data.participant_split(baseline_cycles, test_frac=0.3, seed=SEED)
model_cycles = baseline_cycles.loc[
    baseline_cycles.study_interval.isin(MODEL_INTERVALS)
].copy()
model_frame = data.make_daily_prediction_frame(model_cycles)
model_train_cycles, model_test_cycles = model_split.apply(model_cycles)
model_train_frame, model_test_frame = model_split.apply(model_frame)

assert set(model_train_frame.id).isdisjoint(model_test_frame.id)
assert model_frame.study_interval.isin(MODEL_INTERVALS).all()
assert TARGET in model_frame.columns

# TODO(TIMING_ALIGNMENT): Before joining rolling predictors, settle the cutoff.
# Temperature uses sleep-end day; breathing rate is timestamped at wake time.
# HRV's 18:00 day shift is provisional; daily HR availability is undocumented.
# The exploratory plots include today's readings: do not reuse them blindly.
# Refit/evaluate baselines on these same Interval 1 rows for a fair comparison.
# Keep feature selection inside the training participants; reserve test participants.
print(f"Target: {TARGET}; intervals: {MODEL_INTERVALS}")
print(f"Train: {model_train_frame.id.nunique()} participants, {len(model_train_frame):,} rows")
print(f"Test: {model_test_frame.id.nunique()} participants, {len(model_test_frame):,} rows")
